# 🔬 Eksperimen Gabungan 1 & 2: Repeated LOPO-CV (10x Uji Konsistensi)
**Kelompok Peneliti**: Kelompok A01 — RPL A (Magister Teknik Informatika ITS)  
**Tujuan Eksperimen**: Menguji secara komprehensif performa, kestabilan stokastik, dan konsistensi empiris dari **Tri-Paradigma Data Cleaning (C1: Row Drop vs C2: Project Drop vs C3: Imputasi Cerdas)** yang dikombinasikan dengan 3 Feature Scaling, 4 Imbalance Handling, dan 2 Classifiers dalam matriks **72 Faktorial Pipelines** lintas **10 Independent Random Seeds** (Total: **16.080 Model Folds**).  
**Skema Validasi**: Leave-One-Project-Out Cross-Validation (LOPO-CV, Zero Data Leakage) berulang 10 kali (`[42, 100, 2024, 777, 999, 123, 456, 789, 2025, 2026]`).

## 1. Verifikasi Lingkungan & Hardware GPU RTX 4050

In [ ]:
import sys
import os
from pathlib import Path

src_dir = str(Path("../src").resolve())
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

import check_gpu
check_gpu.check_nvidia_smi()
check_gpu.check_python_environment()
check_gpu.check_xgboost_gpu()

## 2. Ingestion & Komparasi Awal Tri-Paradigma Data (C1, C2, C3)

In [ ]:
from data_loader import load_dataset

df_c1, feat_c1, proj_c1 = load_dataset('C1')
df_c2, feat_c2, proj_c2 = load_dataset('C2')
df_c3, feat_c3, proj_c3 = load_dataset('C3')

print("\n[KOMPARASI AWAL DATASET]:")
print(f"  C1 (Row-Level Drop)     : {len(df_c1):,} baris | {len(proj_c1)} proyek | {df_c1['flaky'].sum()} flaky ({df_c1['flaky'].mean()*100:.2f}%)")
print(f"  C2 (Project-Level Drop) : {len(df_c2):,} baris | {len(proj_c2)} proyek | {df_c2['flaky'].sum()} flaky ({df_c2['flaky'].mean()*100:.2f}%)")
print(f"  C3 (Imputasi Cerdas)    : {len(df_c3):,} baris | {len(proj_c3)} proyek | {df_c3['flaky'].sum()} flaky ({df_c3['flaky'].mean()*100:.2f}%)")

## 3. Eksekusi Repeated LOPO-CV (Full 72 Pipelines x 10 Random Seeds = 16.080 Model Folds)
Menjalankan seluruh 72 kombinasi faktorial (C1/C2/C3 x S1/S2/S3 x B1/B2/B3/B4 x M1/M2) lintas 10 random seeds independen.
- Jika hasil eksekusi (`results/all_seed_pipeline_runs.csv`) sudah tersedia di disk, sel ini langsung memuatnya secara instan.
- Untuk mengeksekusi ulang seluruh 16.080 model folds, ubah `force_rerun = True`.

In [ ]:
import pandas as pd
from pipeline_builder import get_pipeline_specs
from repeated_lopo_runner import run_repeated_lopo_experiments, DEFAULT_SEEDS_10

all_specs = get_pipeline_specs()
runs_file = Path("../results/all_seed_pipeline_runs.csv")

# Parameter kontrol:
force_rerun = False  # Ubah ke True jika ingin menjalankan ulang 16.080 model folds (~3.5 - 4 jam)

if not runs_file.exists() or force_rerun:
    print(f"🚀 Memulai repeated LOPO-CV penuh untuk seluruh {len(all_specs)} pipelines pada 10 seeds...")
    results_full = run_repeated_lopo_experiments(
        pipelines=all_specs,
        seeds=DEFAULT_SEEDS_10,
        output_dir="../results",
        use_gpu=True
    )
    df_runs = results_full["all_runs"]
else:
    print(f"[OK] Hasil eksperimen 10x repetisi ditemukan: {runs_file.resolve()}")
    df_runs = pd.read_csv(runs_file)
    print(f"     Total Eksekusi: {len(df_runs):,} baris evaluasi pipeline ({df_runs['Pipeline_ID'].nunique()} pipelines x {df_runs['Seed'].nunique()} seeds)")

display(df_runs.head(10))

## 4. Analisis Konsistensi & Tabel Stabilitas Empiris
Menghitung Mean ($\mu$), Standar Deviasi ($\sigma$), Coefficient of Variation ($CV = \frac{\sigma}{\mu} \times 100\%$), serta rentang nilai (Min-Max) dan 95% Confidence Interval untuk setiap pipeline dan setiap faktor eksperimen.

In [ ]:
from consistency_analyzer import analyze_pipeline_consistency

df_cons = analyze_pipeline_consistency(
    all_runs_csv="../results/all_seed_pipeline_runs.csv",
    output_dir="../results"
)

cols_show = [
    "Pipeline_ID", "Cleaning", "Scaling", "Imbalance", "Model", "Num_Seeds",
    "Mean_F1_Mean", "Mean_F1_Std", "Mean_F1_CV_%", "Mean_F1_CI95",
    "Mean_Recall_Mean", "Mean_Precision_Mean", "Mean_PR_AUC_Mean"
]
display(df_cons[[c for c in cols_show if c in df_cons.columns]].head(15))

## 5. Uji Signifikansi Statistik (Friedman Test, Nemenyi CD & Kruskal-Wallis)

In [ ]:
from statistical_tests import run_friedman_nemenyi_test, run_cleaning_comparison_test

print("=== UJI STATISTIK FRIEDMAN (72 PIPELINES) ===")
run_friedman_nemenyi_test("../results/summary_metrics_72_pipelines.csv", metric="Mean_F1")

print("\n=== UJI KRUSKAL-WALLIS KOMPARASI TRI-PARADIGMA CLEANING (C1 vs C2 vs C3) ===")
run_cleaning_comparison_test("../results/summary_metrics_72_pipelines.csv")

## 6. Visualisasi Publikasi Ilmiah (300 DPI)
Menghasilkan 6 grafik publikasi standar IEEE/ACM ke direktori `results/plots/` dan menampilkannya di notebook.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from visualizer import generate_all_visualizations

# 1. Generate seluruh 6 gambar resolusi tinggi (300 DPI)
generate_all_visualizations(results_dir="../results", plots_dir="../results/plots")

# 2. Render plot utama langsung di notebook
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(mpimg.imread("../results/plots/01_boxplot_tri_paradigm_cleaning.png"))
axes[0].axis("off")
axes[0].set_title("Plot 1: Komparasi Tri-Paradigma Cleaning", fontweight="bold", fontsize=12)

axes[1].imshow(mpimg.imread("../results/plots/03_top10_consistent_pipelines_ci95.png"))
axes[1].axis("off")
axes[1].set_title("Plot 3: Top 10 Pipeline Konsisten (95% CI)", fontweight="bold", fontsize=12)
plt.tight_layout()
plt.show()

fig2, axes2 = plt.subplots(1, 2, figsize=(16, 6))
axes2[0].imshow(mpimg.imread("../results/plots/04_stability_cv_distribution.png"))
axes2[0].axis("off")
axes2[0].set_title("Plot 4: Uji Stabilitas Empiris (CV%)", fontweight="bold", fontsize=12)

axes2[1].imshow(mpimg.imread("../results/plots/05_f1_vs_recall_tradeoff.png"))
axes2[1].axis("off")
axes2[1].set_title("Plot 5: Trade-off F1 vs Recall (PR-AUC)", fontweight="bold", fontsize=12)
plt.tight_layout()
plt.show()